# ctl-stop-harness on Colab

Runs the stop-mechanism conformance tests against two **synthetic** targets. Nothing here measures a real
system. Allow about five minutes. Colab uses shared CPUs, so read the note at the end before judging timings.

In [ ]:
# 1. Get the code from the public repository (safe to run again)
REPO_URL = "https://github.com/nvns10/ctl-stop-harness"
import os
%cd /content
if not os.path.isdir("ctl-stop-harness"):
    !git clone {REPO_URL} ctl-stop-harness
%cd /content/ctl-stop-harness
!pip -q install -e ".[frameworks]"


In [ ]:
# 2. The environment the timings will come from
import os, platform, sys
print("python", sys.version.split()[0], "| cpus", os.cpu_count(), "|", platform.platform())
!python -m stopharness list-controls

In [ ]:
# 3. Run all three controls against the conforming simulated target (about 10 s)
!python -m stopharness run --mode conformant --no-mapping --out out/conformant | head -16

In [ ]:
# 4. Verify the signed evidence bundle, then tamper with it and verify again
!python -m stopharness verify out/conformant
!echo '{"x":1}' >> out/conformant/events/CTL-STOP-003-t1.jsonl
!python -m stopharness verify out/conformant

In [ ]:
# 5. Seed defects and see which control catches each (about a minute)
from stopharness.controls import load_controls
from stopharness.runner import run_controls
from stopharness.targets import make_target

rows = [("sim_payments", m) for m in ("conformant", "leaky_queue", "slow_under_load", "fail_open", "evidence_loss")]
rows += [("langgraph_payments", m) for m in ("conformant", "stop_ignored", "fail_open")]
print(f"{'target / mode':<40}{'003':<14}{'004':<14}{'005':<14}")
for target, mode in rows:
    r = run_controls(make_target(target, mode), load_controls(), trials=3)
    v = [o.result.value for o in r.outcomes]
    print(f"{target + ' / ' + mode:<40}{v[0]:<14}{v[1]:<14}{v[2]:<14}")

In [ ]:
# 6. The same controls as an Inspect task (the model is a placeholder; the target is what is tested)
!inspect eval inspect_task.py -T target=langgraph_payments -T mode=stop_ignored -T trials=3 --model mockllm/model --max-samples 1 --display plain 2>&1 | tail -12

In [ ]:
# 7. Which framework provisions each result is relevant to (MAS, EU AI Act, IMDA, NIST, ISO)
!python -m stopharness map

## Reading the results on Colab

Expected: the conforming modes PASS, each seeded defect FAILs the control meant to catch it, and
`evidence_loss` is INCONCLUSIVE for 003 and 005.

Timings come from a shared virtual machine. If a conforming mode fails the latency control, that is a
finding about this environment's scheduling jitter, not about the target. Run it again and look at the
reported maximum before drawing a conclusion. Do not raise the 0.5 s reference threshold just to make it pass:
a real threshold needs a traceable source.